# Chapter 13 explore: Continuous Fine-Tuning -- Keeping the Model Current

Interactive companion to `code/chapter_13/continuous_finetune.py`, the book's final chapter. Splits this book's real archive by a chronological cutoff, trains a "current" model, continues training on the "new" reports, and runs Chapter 12's comparison to decide whether the update is actually worth deploying. Running the full training takes roughly 20-25 minutes on CPU -- this notebook re-runs the comparison against the checkpoints a full run already produced, rather than retraining from scratch.

In [1]:
import sys
sys.path.insert(0, "../code/chapter_01")
sys.path.insert(0, "../code/chapter_02")
sys.path.insert(0, "../code/chapter_03")
sys.path.insert(0, "../code/chapter_05")
sys.path.insert(0, "../code/chapter_06")
sys.path.insert(0, "../code/chapter_07")
sys.path.insert(0, "../code/chapter_08")
sys.path.insert(0, "../code/chapter_09")
sys.path.insert(0, "../code/chapter_10")
sys.path.insert(0, "../code/chapter_11")
sys.path.insert(0, "../code/chapter_12")
sys.path.insert(0, "../code/chapter_13")

from eval_finetuned_model import build_held_out_eval_set
from detect_model_drift import load_version, summarize_version, compare_versions
from continuous_finetune import RUNS_DIR, split_archive_by_cutoff, build_examples

current_reports, new_reports = split_archive_by_cutoff()
print(f"'Currently in production': {len(current_reports)} reports, {len(build_examples(current_reports))} examples")
print(f"New reports just arrived:  {len(new_reports)} reports, {len(build_examples(new_reports))} examples")

'Currently in production': 57 reports, 490 examples


New reports just arrived:  17 reports, 179 examples


Compare the current model (`checkpoint_2`) against the updated one (`checkpoint_4`) from a real run. Uses Chapter 12's `load_version` (not Chapter 8's `load_checkpoint`) deliberately -- `load_checkpoint` leaves the model in train mode for resuming training, which makes evaluation noisy for no reason.

In [2]:
runs = sorted(RUNS_DIR.glob("run_*"))
run_dir = runs[-1]
print(f"Using run: {run_dir.name}")

eval_set = build_held_out_eval_set()

lora_current, tokenizer = load_version(run_dir / "checkpoint_2")
summary_current = summarize_version(lora_current, tokenizer, eval_set)

lora_updated, _ = load_version(run_dir / "checkpoint_4")
summary_updated = summarize_version(lora_updated, tokenizer, eval_set)

print("Current model:", summary_current)
print("Updated model:", summary_updated)
print("Direction:    ", compare_versions(summary_current, summary_updated))

Using run: run_20260817_013444


`torch_dtype` is deprecated! Use `dtype` instead!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Current model: {'exact_match': 0, 'avg_overlap': 0.46428571428571425, 'perplexity': 25.916617420628782}
Updated model: {'exact_match': 0, 'avg_overlap': 0.125, 'perplexity': 25.727166793932724}
Direction:     {'exact_match': 'unchanged', 'avg_overlap': 'regressed', 'perplexity': 'improved'}
